# 06 — Kizomba Batida Tracker Check

Phase 25 — visual + audio validation that the new kizomba-only beat tracker (`_track_kizomba_batida`, gated via `dance_style="kizomba"`) lands its green dashed beat lines on the **kick** transients for kizomba tracks, while leaving bachata untouched.

For each track you get **two** interactive timelines (same widget as notebook 05 — horizontal scrollbar + play button + moving red cursor):

1. **OLD** — `analyze(audio, dance_style=None)`, the historical `librosa.beat.beat_track` path.
2. **NEW** — `analyze(audio, dance_style=style)`, the batida tracker (only different for kizomba/urban_kiz).

Each timeline is decorated with two extra layers:

- **Numbered onset labels** above the wave, one per onset, reset to `1` after every measure (`M1`, `M2`, ...). Color-coded by beat proximity:
  - **orange** = onset within ~60 ms of a downbeat ("the 1")
  - **green**  = onset within ~60 ms of any beat
  - **grey**   = onset that is between beats
- **Blue ▼ markers** above the wave on the NEW timeline only — these are the **kick-band onsets** the new tracker actually picks from. The green dashed beats should land on the blue triangles, not necessarily on the thin red full-spectrum onset lines.
- **Purple ✕ markers** above the wave on the NEW timeline — your own *perceived step positions*, taken from `LISTENING_NOTES` in the config cell. Compare green dashed beats vs purple ✕: where they disagree is exactly where the tracker is missing the dance feel.

Edit `TRACKS` below to control the subset.

In [ ]:
%matplotlib inline
from pathlib import Path

import librosa
import matplotlib.pyplot as plt
import numpy as np
from IPython.display import HTML, display
from scipy.signal import butter, filtfilt

from rytmi import analyze, load_audio
from rytmi import viz as rytmi_viz
from rytmi.dsp import _KIZOMBA_BATIDA_HOP, _KIZOMBA_BATIDA_LPF_HZ
from rytmi.eval.listening_notes import find_notes_for_path, load_notes, notes_to_times

EVAL = Path("../data/songs/eval_set")

# All five kizomba tracks that have tap takes under data/eval/taps/.
# Bachata kept as a non-kizomba sanity check.
TRACKS: list[tuple[str, Path]] = [
    ("kizomba", EVAL / "kizomba" / "Baila_Kizomba_Amor [XG11YxMWgaI].mp3"),
    ("kizomba", EVAL / "kizomba" / "Filomena_Maricoa_-_Teu_Toque_Official_Video [sKbWzD0mt6I].mp3"),
    ("kizomba", EVAL / "kizomba" / "MIKA_MENDES_-_MAGICO_2011 [ZM1GnUImrPw].mp3"),
    ("kizomba", EVAL / "kizomba" / "Charbel_-_E_Magia_Official_Video_4K [QkfyDj8aJRM].mp3"),
    ("kizomba", EVAL / "kizomba" / "Criola [kCOie6jQXag].mp3"),
    ("bachata", EVAL / "bachata" / "Romeo_Santos_-_Propuesta_Indecente_Official_Video [QFs3PIZb3js].mp3"),
]

PIXELS_PER_SECOND = 80
BEAT_TOLERANCE_S = 0.06

# Hand-marked perceived step positions live in
# data/eval/kizomba_listening_notes.yaml. Notation is documented there.
LISTENING_NOTES_PATH = Path("../data/eval/kizomba_listening_notes.yaml")
LISTENING_NOTES = load_notes(LISTENING_NOTES_PATH)


## Helpers — kick-band onsets, decorated `plot_timeline`, render function

We monkey-patch `rytmi.viz.plot_timeline` with a wrapper that adds the two extra layers (numbered onsets + optional kick triangles) before the figure is rasterized for the interactive widget. The wrapper reads the current track's kick times from a module-level slot set per `render()` call.

In [ ]:
def kick_onset_times(audio) -> np.ndarray:
    nyq = 0.5 * audio.sr
    b, a = butter(4, _KIZOMBA_BATIDA_LPF_HZ / nyq, btype="low")
    y_lpf = filtfilt(b, a, audio.samples).astype(np.float32, copy=False)
    # fmin=20 + n_mels=8 essential: librosa default n_mels=128 has no
    # mel filter below ~80 Hz, so a 150 Hz LPF input would silently
    # produce an all-zero envelope (Phase 28 P5d). Mirrors
    # src/rytmi/dsp.py::_track_kizomba_batida.
    env = librosa.onset.onset_strength(
        y=y_lpf, sr=audio.sr,
        fmin=20.0, fmax=_KIZOMBA_BATIDA_LPF_HZ * 1.5, n_mels=8,
        aggregate=np.median,
    )
    frames = librosa.onset.onset_detect(onset_envelope=env, sr=audio.sr)
    return librosa.frames_to_time(frames, sr=audio.sr, hop_length=_KIZOMBA_BATIDA_HOP)


_OVERLAY_KICKS: np.ndarray | None = None
_OVERLAY_STEPS: np.ndarray | None = None


_ORIG_PLOT_TIMELINE = rytmi_viz.plot_timeline


def _decorated_plot_timeline(analysis, **kwargs):
    fig = _ORIG_PLOT_TIMELINE(analysis, **kwargs)
    if not fig.axes:
        return fig
    ax = fig.axes[0]
    onset_times = np.asarray(analysis.onsets.times)
    beat_times = np.asarray(analysis.beats.times)
    downbeats = analysis.downbeats
    downbeat_times = np.asarray(downbeats) if downbeats is not None else np.array([])
    bpm = max(int(getattr(analysis, "beats_per_measure", 4) or 4), 1)
    amp_max = float(np.max(np.abs(analysis.audio.samples))) or 1.0
    if len(downbeat_times) > 0:
        boundaries = downbeat_times
    elif len(beat_times) > 0:
        offset = analysis.downbeat_offset or 0
        boundaries = beat_times[offset::bpm]
    else:
        boundaries = np.array([])
    y_onset_num = amp_max * 1.30
    if len(onset_times) > 0:
        m_idx = np.searchsorted(boundaries, onset_times, side="right") - 1
        counter: dict[int, int] = {}
        for t, m in zip(onset_times, m_idx, strict=False):
            counter[int(m)] = counter.get(int(m), 0) + 1
            n = counter[int(m)]
            if len(downbeat_times) and np.min(np.abs(downbeat_times - t)) <= BEAT_TOLERANCE_S:
                color, weight = "darkorange", "bold"
            elif len(beat_times) and np.min(np.abs(beat_times - t)) <= BEAT_TOLERANCE_S:
                color, weight = "green", "bold"
            else:
                color, weight = "#888", "normal"
            ax.text(t, y_onset_num, str(n), fontsize=7, color=color,
                    fontweight=weight, ha="center", va="bottom", zorder=4)
    if _OVERLAY_KICKS is not None and len(_OVERLAY_KICKS):
        y_kick = amp_max * 1.05
        ax.scatter(_OVERLAY_KICKS, np.full(len(_OVERLAY_KICKS), y_kick),
                   marker="v", color="#1f6feb", s=42, zorder=5,
                   clip_on=False, label="kick-band onsets")
    if _OVERLAY_STEPS is not None and len(_OVERLAY_STEPS):
        y_step = amp_max * 1.20
        ax.scatter(_OVERLAY_STEPS, np.full(len(_OVERLAY_STEPS), y_step),
                   marker="X", color="#9b59d0", s=70, zorder=6,
                   clip_on=False, label="perceived step (listening note)")
    ymin, ymax = ax.get_ylim()
    if y_onset_num + amp_max * 0.05 > ymax:
        ax.set_ylim(ymin, y_onset_num + amp_max * 0.10)
    return fig


rytmi_viz.plot_timeline = _decorated_plot_timeline


def render(style: str, path: Path) -> None:
    global _OVERLAY_KICKS, _OVERLAY_STEPS
    name = path.stem.split(" [")[0]
    print(f"\n========== {name}  (dance_style={style!r}) ==========")
    audio = load_audio(path)
    old = analyze(audio, dance_style=None)
    print(f"  OLD  tempo={old.tempo:6.1f} BPM | beats={len(old.beats.times):4d} | onsets={len(old.onsets.times)}")
    _OVERLAY_KICKS = None
    display(HTML(rytmi_viz.interactive_timeline(
        old, title=f"{name} — OLD (librosa default)",
        pixels_per_second=PIXELS_PER_SECOND,
    )))
    plt.close("all")
    new = analyze(audio, dance_style=style)
    print(f"  NEW  tempo={new.tempo:6.1f} BPM | beats={len(new.beats.times):4d} | onsets={len(new.onsets.times)}")
    _OVERLAY_KICKS = (
        kick_onset_times(audio) if style in {"kizomba", "urban_kiz"} else None
    )
    if _OVERLAY_KICKS is not None:
        print(f"  kick-band onsets (▼ blue overlay): {len(_OVERLAY_KICKS)}")
    track_notes = find_notes_for_path(path, LISTENING_NOTES)
    if track_notes is not None:
        _OVERLAY_STEPS = notes_to_times(list(track_notes.notes), new)
        print(f"  perceived steps (✕ purple overlay) from {track_notes.stem!r}: "
              f"{len(_OVERLAY_STEPS)} / {len(track_notes.notes)} resolved")
    else:
        _OVERLAY_STEPS = None
    display(HTML(rytmi_viz.interactive_timeline(
        new, title=f"{name} — NEW (dance_style={style!r})",
        pixels_per_second=PIXELS_PER_SECOND,
    )))
    plt.close("all")
    _OVERLAY_KICKS = None
    _OVERLAY_STEPS = None


## Run the comparison

For each track in `TRACKS`: OLD interactive timeline → NEW interactive timeline. Each has its own play button + scrolling cursor. Compare the green dashed beat lines against the **blue ▼ kick markers** on the NEW timeline (kizomba only).

In [ ]:
for style, path in TRACKS:
    render(style, path)

## Phase 27 — multi-band onset diagnostic

The Phase 25 kizomba tracker uses a **low-pass at 150 Hz** to isolate the kick. User feedback: beats land *off by a small constant* relative to perceived steps. Hypothesis: the perceived step is cued by something brighter than just the kick (snap, clave, rim) — so an onset envelope from a higher band might land closer to the ✕ listening notes.

For each kizomba track with listening notes, this cell plots **four candidate onset bands** stacked above the wave, restricted to the time window the listening notes cover, with the purple ✕ marks for direct visual comparison:

- **low** &nbsp;&nbsp; 0–150 Hz (current default)
- **mid** &nbsp;&nbsp; 200–800 Hz (snap, mid percussion)
- **high** &nbsp; 1.5–6 kHz (clave, clap, rim)
- **perc** &nbsp; HPSS percussive component (broadband, harmonics removed)

If one band consistently hugs the ✕ marks better than `low`, that's the candidate to ship in Phase 27.P3.

In [ ]:
def _band_onsets(audio, band: tuple) -> np.ndarray:
    """Onset times computed from one of four input bands.

    band kinds: ("low", hi), ("band", lo, hi), ("perc", None).
    Mirrors scripts/eval_kizomba_beats.py::_filter_audio so visual and
    numeric pictures stay in sync.
    """
    sr = audio.sr
    nyq = 0.5 * sr
    kind = band[0]
    if kind == "low":
        b, a = butter(4, band[1] / nyq, btype="low")
        y = filtfilt(b, a, audio.samples).astype(np.float32, copy=False)
        # fmin=20 + n_mels=8 essential for low-band envelopes; default
        # mel filterbank has no bins below ~80 Hz → all-zero envelope
        # (Phase 28 P5d).
        env = librosa.onset.onset_strength(
            y=y, sr=sr, fmin=20.0, fmax=float(band[1]) * 1.5, n_mels=8,
            aggregate=np.median, hop_length=_KIZOMBA_BATIDA_HOP,
        )
        frames = librosa.onset.onset_detect(onset_envelope=env, sr=sr)
        return librosa.frames_to_time(frames, sr=sr, hop_length=_KIZOMBA_BATIDA_HOP)
    elif kind == "band":
        b, a = butter(4, [band[1] / nyq, band[2] / nyq], btype="band")
        y = filtfilt(b, a, audio.samples).astype(np.float32, copy=False)
        fmax = None
    elif kind == "perc":
        _, y = librosa.effects.hpss(audio.samples.astype(np.float32, copy=False))
        fmax = None
    else:
        raise ValueError(band)
    env = librosa.onset.onset_strength(
        y=y, sr=sr, fmax=fmax, aggregate=np.median, hop_length=_KIZOMBA_BATIDA_HOP
    )
    frames = librosa.onset.onset_detect(onset_envelope=env, sr=sr)
    return librosa.frames_to_time(frames, sr=sr, hop_length=_KIZOMBA_BATIDA_HOP)


_DIAG_BANDS: list[tuple[str, tuple, str]] = [
    ("low",  ("low",  150.0),          "#1f6feb"),  # current default
    ("mid",  ("band", 200.0, 800.0),   "#2ca02c"),
    ("high", ("band", 1500.0, 6000.0), "#d62728"),
    ("perc", ("perc", None),           "#9467bd"),
]


def diag_band_overlay(path: Path, pad_s: float = 0.5) -> None:
    """Static plot: wave + 4 band-onset rows + ✕ listening notes."""
    audio = load_audio(path)
    analysis = analyze(audio, dance_style="kizomba")
    track_notes = find_notes_for_path(path, LISTENING_NOTES)
    if track_notes is None:
        print(f"  (skipping {path.stem} — no listening notes)")
        return
    notes_t = notes_to_times(list(track_notes.notes), analysis)
    if not notes_t.size:
        print(f"  (skipping {path.stem} — listening notes did not resolve)")
        return
    t0, t1 = float(notes_t.min()) - pad_s, float(notes_t.max()) + pad_s

    name = path.stem.split(" [")[0]
    fig, ax = plt.subplots(figsize=(14, 4.2))
    sr = audio.sr
    n0, n1 = max(0, int(t0 * sr)), min(audio.samples.size, int(t1 * sr))
    ts = np.linspace(t0, t1, n1 - n0, endpoint=False)
    ax.plot(ts, audio.samples[n0:n1], color="#bbb", lw=0.6, zorder=1)
    amp_max = float(np.max(np.abs(audio.samples[n0:n1]))) or 1.0

    # Stack one onset row per band above the wave.
    for i, (label, band, color) in enumerate(_DIAG_BANDS):
        ot = _band_onsets(audio, band)
        ot_w = ot[(ot >= t0) & (ot <= t1)]
        y = amp_max * (1.10 + 0.18 * i)
        ax.scatter(ot_w, np.full(len(ot_w), y), marker="v", s=36,
                   color=color, label=f"{label} ({len(ot_w)})", zorder=4)
        ax.text(t0 - 0.05, y, label, fontsize=9, color=color,
                ha="right", va="center")

    # ✕ listening notes on top.
    notes_w = notes_t[(notes_t >= t0) & (notes_t <= t1)]
    y_steps = amp_max * (1.10 + 0.18 * len(_DIAG_BANDS))
    ax.scatter(notes_w, np.full(len(notes_w), y_steps), marker="X", s=80,
               color="#9b59d0", zorder=5, label=f"listening notes ({len(notes_w)})")
    ax.text(t0 - 0.05, y_steps, "✕ steps", fontsize=9, color="#9b59d0",
            ha="right", va="center", fontweight="bold")
    # Faint vertical guide at every ✕.
    for t in notes_w:
        ax.axvline(t, color="#9b59d0", lw=0.5, alpha=0.25, zorder=2)

    ax.set_xlim(t0, t1)
    ax.set_title(f"{name} — band onsets vs listening notes")
    ax.set_xlabel("time (s)")
    ax.set_yticks([])
    ax.legend(loc="lower right", fontsize=8, ncol=5)
    plt.tight_layout()
    plt.show()


for _, p in TRACKS:
    if "kizomba" not in str(p):
        continue
    print(f"\n----- {p.stem.split(' [')[0]} -----")
    diag_band_overlay(p)


## Phase 28 P1 — onset-strength envelope diagnostic

The Phase 27 sweep showed that swapping the input band changed almost nothing — the **`mid` config was bitwise identical to `low`**. Reason: the same N peaks get picked regardless of band because the picker's `wait_ms` floor dominates. Before trying a multi-band *sum* envelope (Phase 28 P4), we want to see what the picker is actually looking at.

For each kizomba track, this cell plots the **onset-strength envelope** in 4 stacked panels:

- **low** &nbsp;&nbsp; 0–150 Hz (current Phase 25 default)
- **mid** &nbsp;&nbsp; 200–800 Hz (snap, mid percussion)
- **high** &nbsp; 1.5–6 kHz (clave, clap, rim)
- **perc** &nbsp; HPSS percussive component (broadband, harmonics removed)

Overlaid on each: production picked beats (green dashed) and ✕ listening notes (purple). Read by eye: do all four envelopes peak at the same times, or do bands disagree? If they agree → the multi-band sum will be a wash and we should make tap data the only Phase 28 deliverable. If they disagree → summing them is the right next step.

In [ ]:
def _band_envelope(audio, band: tuple):
    """Return (envelope, times) for one input band.

    Reuses the band-filter logic from `_band_onsets` above so visual and
    numeric pictures stay aligned. The envelope is the curve the picker
    actually sees.
    """
    sr = audio.sr
    nyq = 0.5 * sr
    kind = band[0]
    if kind == "low":
        b, a = butter(4, band[1] / nyq, btype="low")
        y = filtfilt(b, a, audio.samples).astype(np.float32, copy=False)
        # fmin=20 + n_mels=8 essential for low-band envelopes; default
        # mel filterbank has no bins below ~80 Hz → all-zero envelope
        # (Phase 28 P5d). Without these args, the "low" panel below
        # would be flat zero everywhere, not the truth about the kick.
        env = librosa.onset.onset_strength(
            y=y, sr=sr, fmin=20.0, fmax=float(band[1]) * 1.5, n_mels=8,
            aggregate=np.median, hop_length=_KIZOMBA_BATIDA_HOP,
        )
        times = librosa.frames_to_time(
            np.arange(env.size), sr=sr, hop_length=_KIZOMBA_BATIDA_HOP
        )
        return env, times
    elif kind == "band":
        b, a = butter(4, [band[1] / nyq, band[2] / nyq], btype="band")
        y = filtfilt(b, a, audio.samples).astype(np.float32, copy=False)
        fmax = None
    elif kind == "perc":
        _, y = librosa.effects.hpss(audio.samples.astype(np.float32, copy=False))
        fmax = None
    else:
        raise ValueError(band)
    env = librosa.onset.onset_strength(
        y=y, sr=sr, fmax=fmax, aggregate=np.median, hop_length=_KIZOMBA_BATIDA_HOP
    )
    times = librosa.frames_to_time(
        np.arange(env.size), sr=sr, hop_length=_KIZOMBA_BATIDA_HOP
    )
    return env, times


def diag_envelope_stack(path: Path, pad_s: float = 0.5) -> None:
    """4-panel onset-envelope view, restricted to the listening-note window."""
    audio = load_audio(path)
    analysis = analyze(audio, dance_style="kizomba")
    track_notes = find_notes_for_path(path, LISTENING_NOTES)
    if track_notes is None:
        print(f"  (skipping {path.stem} — no listening notes)")
        return
    notes_t = notes_to_times(list(track_notes.notes), analysis)
    if not notes_t.size:
        print(f"  (skipping {path.stem} — listening notes did not resolve)")
        return
    t0, t1 = float(notes_t.min()) - pad_s, float(notes_t.max()) + pad_s
    beats_t = np.asarray(analysis.beats.times)
    beats_w = beats_t[(beats_t >= t0) & (beats_t <= t1)]
    notes_w = notes_t[(notes_t >= t0) & (notes_t <= t1)]

    name = path.stem.split(" [")[0]
    fig, axes = plt.subplots(
        len(_DIAG_BANDS), 1, figsize=(14, 2.0 * len(_DIAG_BANDS)),
        sharex=True,
    )
    for ax, (label, band, color) in zip(axes, _DIAG_BANDS, strict=True):
        env, ts = _band_envelope(audio, band)
        mask = (ts >= t0) & (ts <= t1)
        env_w = env[mask]
        ts_w = ts[mask]
        # Per-panel max-normalize so panel heights are comparable.
        env_norm = env_w / (env_w.max() if env_w.size and env_w.max() > 0 else 1.0)
        ax.fill_between(ts_w, 0, env_norm, color=color, alpha=0.35,
                        linewidth=0.0)
        ax.plot(ts_w, env_norm, color=color, lw=0.8)
        # Picked beats (production).
        for t in beats_w:
            ax.axvline(t, color="green", lw=0.8, ls="--", alpha=0.55,
                       zorder=3)
        # ✕ listening notes.
        for t in notes_w:
            ax.axvline(t, color="#9b59d0", lw=1.0, alpha=0.7, zorder=4)
        ax.set_ylim(0, 1.1)
        ax.set_yticks([])
        ax.set_ylabel(label, rotation=0, ha="right", va="center",
                      fontsize=10, color=color, fontweight="bold")
    axes[-1].set_xlabel("time (s)")
    axes[0].set_title(
        f"{name} — onset-strength envelope per band  "
        f"(green dashed = picked beats, purple = listening notes)"
    )
    axes[0].set_xlim(t0, t1)
    plt.tight_layout()
    plt.show()


for _, p in TRACKS:
    if "kizomba" not in str(p):
        continue
    print(f"\n----- {p.stem.split(' [')[0]} -----")
    diag_envelope_stack(p)
